# Azure Machine Learning SDK v2 - Complete Integration with Databricks

This notebook demonstrates end-to-end integration between Azure Databricks and Azure Machine Learning using the **AzureML SDK v2**.

## 📚 Contents

1. **Install Dependencies** - azure-ai-ml and azure-identity
2. **Configure AzureML Client** - Authentication and workspace connection
3. **Submit Command Job** - Run training scripts from Databricks to AzureML
4. **Register Models** - Register models from DBFS to AzureML
5. **Trigger AutoML Job** - Automated machine learning with AzureML
6. **Log Metrics** - Track experiments and metrics from Databricks
7. **AzureML Pipeline with Databricks** - Orchestrate Databricks jobs in AzureML pipelines

## 🔑 Prerequisites

- Azure subscription with proper permissions
- Azure Machine Learning workspace
- Azure Databricks workspace
- Service principal or managed identity for authentication

---

## 1. Install Dependencies

Install the required Azure ML SDK v2 packages:
- `azure-ai-ml`: Main SDK for Azure Machine Learning
- `azure-identity`: Authentication library for Azure services

In [ ]:
%pip install azure-ai-ml azure-identity --quiet

## 2. Configure AzureML Client

Establish connection to Azure Machine Learning workspace using DefaultAzureCredential.

**Authentication Flow:**
- Attempts multiple authentication methods in order
- Works with managed identity, service principal, Azure CLI, and more
- Secure and recommended for production environments

**Replace the placeholder values below with your Azure ML workspace details:**

In [ ]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

# Azure ML Workspace Configuration
# TODO: Replace these placeholders with your actual values
subscription_id = "<SUBSCRIPTION_ID>"  # Your Azure subscription ID
resource_group = "<RESOURCE_GROUP>"    # Resource group containing AzureML workspace
workspace_name = "<WORKSPACE_NAME>"    # Azure ML workspace name

# Create authenticated MLClient
credential = DefaultAzureCredential()
ml_client = MLClient(
    credential=credential,
    subscription_id=subscription_id,
    resource_group_name=resource_group,
    workspace_name=workspace_name
)

print(f"✅ Connected to Azure ML workspace: {workspace_name}")
print(f"   Resource Group: {resource_group}")
print(f"   Subscription: {subscription_id}")

## 3. Submit a Command Job from Databricks to AzureML

Submit a training job to Azure ML from Databricks. This allows you to:
- Leverage AzureML compute for training
- Use AzureML experiment tracking
- Access AzureML datasets and environments

**Key Components:**
- **Command**: Python script to execute
- **Environment**: Docker image or conda environment
- **Compute**: AzureML compute cluster to run on
- **Code**: Local directory containing training scripts

In [ ]:
from azure.ai.ml import command
from azure.ai.ml.entities import Environment

# Define the training command job
job = command(
    code="./src",  # Local directory containing your training script
    command="python train.py --learning_rate 0.01 --batch_size 32",
    environment="AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest",  # Curated environment
    compute="<COMPUTE_CLUSTER_NAME>",  # Replace with your compute cluster name
    experiment_name="databricks-to-azureml",
    display_name="training-from-databricks",
    description="Training job submitted from Databricks to AzureML"
)

# Submit the job
returned_job = ml_client.jobs.create_or_update(job)

print(f"✅ Job submitted successfully!")
print(f"   Job Name: {returned_job.name}")
print(f"   Job ID: {returned_job.id}")
print(f"   Status: {returned_job.status}")
print(f"   Studio URL: {returned_job.studio_url}")

### Optional: Monitor Job Status

Stream job logs and wait for completion:

In [ ]:
# Stream job logs (optional)
# ml_client.jobs.stream(returned_job.name)

## 4. Register a Model in AzureML

Register a trained model from DBFS (Databricks File System) to Azure ML Model Registry.

**Benefits:**
- Centralized model versioning
- Model lineage tracking
- Easy deployment to various endpoints
- Model sharing across teams

**Example:** Registering a scikit-learn pickle model

In [ ]:
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes

# Model file location on DBFS
# You can train and save model to DBFS first, then register it
model_path = "dbfs:/mnt/models/model.pkl"  # Update with your model path

# Copy model from DBFS to local file system for registration
# Note: In practice, you might want to use a temporary location
local_model_path = "/tmp/model.pkl"
dbutils.fs.cp(model_path, f"file://{local_model_path}")

# Create Model entity
model = Model(
    path=local_model_path,
    type=AssetTypes.CUSTOM_MODEL,  # Can be MLFLOW_MODEL, TRITON_MODEL, etc.
    name="databricks-trained-model",
    description="Model trained in Databricks and registered in AzureML",
    tags={
        "framework": "scikit-learn",
        "trained_on": "databricks",
        "use_case": "classification"
    }
)

# Register the model
registered_model = ml_client.models.create_or_update(model)

print(f"✅ Model registered successfully!")
print(f"   Model Name: {registered_model.name}")
print(f"   Model Version: {registered_model.version}")
print(f"   Model ID: {registered_model.id}")

## 5. Trigger an AzureML AutoML Job

Launch an automated machine learning (AutoML) experiment for classification.

**AutoML automatically:**
- Tests multiple algorithms
- Performs hyperparameter tuning
- Handles feature engineering
- Selects the best model based on metrics

**Data Requirements:**
- Data should be in MLTable format or registered as AzureML dataset
- Target column must be specified
- Training and validation splits can be defined

In [ ]:
from azure.ai.ml import automl
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml import Input

# Define training data (MLTable format)
# Replace with your data asset name or path
training_data = Input(
    type=AssetTypes.MLTABLE,
    path="azureml:<DATA_ASSET_NAME>:<VERSION>"  # e.g., "azureml:credit-card:1"
)

# Create AutoML classification job
classification_job = automl.classification(
    compute="<COMPUTE_CLUSTER_NAME>",  # Replace with your compute cluster
    experiment_name="automl-classification-databricks",
    training_data=training_data,
    target_column_name="<TARGET_COLUMN>",  # e.g., "label" or "is_fraud"
    primary_metric="accuracy",  # Options: accuracy, AUC_weighted, precision, recall
    n_cross_validations=5,
    enable_model_explainability=True,
    tags={
        "initiated_from": "databricks",
        "project": "ml-integration"
    }
)

# Configure training settings
classification_job.set_limits(
    timeout_minutes=60,
    trial_timeout_minutes=20,
    max_trials=20,
    max_concurrent_trials=4,
    enable_early_termination=True
)

# Submit AutoML job
automl_job = ml_client.jobs.create_or_update(classification_job)

print(f"✅ AutoML job submitted successfully!")
print(f"   Job Name: {automl_job.name}")
print(f"   Experiment: {automl_job.experiment_name}")
print(f"   Status: {automl_job.status}")
print(f"   Studio URL: {automl_job.studio_url}")

## 6. Log Metrics to AzureML from Databricks

Track experiments and log metrics from Databricks to Azure ML.

**Use Cases:**
- Track model training metrics
- Compare multiple experiment runs
- Store hyperparameters and results
- Build experiment history

**Note:** This uses MLflow integration with AzureML tracking.

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

# Set MLflow tracking URI to AzureML workspace
# Format: azureml://<subscription_id>/<resource_group>/<workspace_name>
mlflow_tracking_uri = f"azureml://{subscription_id}/{resource_group}/{workspace_name}"
mlflow.set_tracking_uri(mlflow_tracking_uri)

# Create or get experiment
experiment_name = "databricks-training-experiments"
mlflow.set_experiment(experiment_name)

# Start a new run and log metrics
with mlflow.start_run(run_name="training-run-001") as run:
    # Log parameters
    mlflow.log_param("learning_rate", 0.01)
    mlflow.log_param("batch_size", 32)
    mlflow.log_param("epochs", 10)
    mlflow.log_param("optimizer", "adam")
    
    # Simulate training and log metrics
    # In real scenario, these would come from your training loop
    mlflow.log_metric("accuracy", 0.95)
    mlflow.log_metric("loss", 0.15)
    mlflow.log_metric("precision", 0.93)
    mlflow.log_metric("recall", 0.92)
    mlflow.log_metric("f1_score", 0.925)
    
    # Log metrics over multiple steps (e.g., epochs)
    for epoch in range(1, 11):
        mlflow.log_metric("train_loss", 0.5 / epoch, step=epoch)
        mlflow.log_metric("val_accuracy", 0.8 + (0.1 * epoch / 10), step=epoch)
    
    # Log tags
    mlflow.set_tags({
        "framework": "scikit-learn",
        "model_type": "random_forest",
        "environment": "databricks"
    })
    
    run_id = run.info.run_id
    
print(f"✅ Metrics logged successfully to AzureML!")
print(f"   Run ID: {run_id}")
print(f"   Experiment: {experiment_name}")
print(f"   View in AzureML Studio: https://ml.azure.com/runs/{run_id}")

### View Logged Metrics

Query and display logged metrics from the run:

In [ ]:
# Retrieve run details
mlflow_client = MlflowClient()
run_data = mlflow_client.get_run(run_id)

print("📊 Run Metrics:")
for key, value in run_data.data.metrics.items():
    print(f"   {key}: {value}")

print("\n⚙️ Run Parameters:")
for key, value in run_data.data.params.items():
    print(f"   {key}: {value}")

## 7. Add an AzureML Pipeline Step that Calls a Databricks Job

Create an AzureML pipeline that orchestrates Databricks job execution.

**Use Cases:**
- Hybrid ML pipelines spanning Databricks and AzureML
- Data processing in Databricks + training in AzureML
- Orchestrate complex workflows across platforms

**Prerequisites:**
- Databricks job must be created in advance
- Databricks compute configured in AzureML
- Proper authentication between services

In [ ]:
from azure.ai.ml.entities import PipelineJob
from azure.ai.ml import dsl
from azure.ai.ml.entities import CommandComponent

# Note: Azure ML SDK v2 doesn't have built-in DatabricksJobTask component
# This demonstrates using a command component that triggers Databricks REST API

# Step 1: Create a command component that calls Databricks API
databricks_trigger_component = CommandComponent(
    name="trigger_databricks_job",
    display_name="Trigger Databricks Job",
    description="Triggers a Databricks job via REST API",
    code="./src",  # Directory with script to call Databricks API
    command="python trigger_databricks.py --job_id ${{inputs.databricks_job_id}} --token ${{inputs.databricks_token}}",
    environment="AzureML-sklearn-1.0-ubuntu20.04-py38-cpu@latest",
    inputs={
        "databricks_job_id": {"type": "string"},
        "databricks_token": {"type": "string"}
    }
)

# Step 2: Define pipeline using DSL
@dsl.pipeline(
    name="databricks_integration_pipeline",
    description="Pipeline that integrates Databricks job execution",
    default_compute="<COMPUTE_CLUSTER_NAME>"  # Replace with your compute
)
def databricks_pipeline(databricks_job_id: str, databricks_token: str):
    """
    Pipeline definition with Databricks job trigger step.
    
    Args:
        databricks_job_id: The Databricks job ID to trigger
        databricks_token: Authentication token for Databricks
    """
    # Step: Trigger Databricks job
    databricks_step = databricks_trigger_component(
        databricks_job_id=databricks_job_id,
        databricks_token=databricks_token
    )
    databricks_step.display_name = "Execute Databricks Data Processing"
    
    return {}

# Step 3: Create pipeline instance
pipeline_job = databricks_pipeline(
    databricks_job_id="<DATABRICKS_JOB_ID>",  # Replace with your job ID
    databricks_token="<DATABRICKS_TOKEN>"      # Use Key Vault or secure string
)

# Configure pipeline settings
pipeline_job.settings.default_compute = "<COMPUTE_CLUSTER_NAME>"
pipeline_job.tags = {
    "pipeline_type": "hybrid",
    "integrates_with": "databricks"
}

# Submit pipeline
pipeline_run = ml_client.jobs.create_or_update(pipeline_job)

print(f"✅ Pipeline submitted successfully!")
print(f"   Pipeline Name: {pipeline_run.name}")
print(f"   Pipeline ID: {pipeline_run.id}")
print(f"   Status: {pipeline_run.status}")
print(f"   Studio URL: {pipeline_run.studio_url}")

### Alternative: Direct Databricks REST API Integration

For more direct control, you can use Databricks REST API to trigger jobs:

In [ ]:
import requests
import time

def trigger_databricks_job(databricks_instance: str, job_id: str, token: str):
    """
    Trigger a Databricks job and wait for completion.
    
    Args:
        databricks_instance: Databricks workspace URL (e.g., 'adb-123456789.azuredatabricks.net')
        job_id: Databricks job ID
        token: Personal access token or service principal token
    
    Returns:
        run_id: The run ID of the triggered job
    """
    # Trigger job
    url = f"https://{databricks_instance}/api/2.1/jobs/run-now"
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }
    payload = {"job_id": job_id}
    
    response = requests.post(url, json=payload, headers=headers)
    response.raise_for_status()
    
    run_id = response.json()["run_id"]
    print(f"✅ Databricks job triggered: Run ID {run_id}")
    
    # Optional: Poll for completion
    status_url = f"https://{databricks_instance}/api/2.1/jobs/runs/get"
    while True:
        status_response = requests.get(
            status_url,
            params={"run_id": run_id},
            headers=headers
        )
        status_response.raise_for_status()
        
        state = status_response.json()["state"]
        life_cycle_state = state["life_cycle_state"]
        
        print(f"   Job status: {life_cycle_state}")
        
        if life_cycle_state in ["TERMINATED", "SKIPPED", "INTERNAL_ERROR"]:
            result_state = state.get("result_state", "UNKNOWN")
            print(f"   Job completed with result: {result_state}")
            break
        
        time.sleep(30)  # Poll every 30 seconds
    
    return run_id

# Example usage (commented out - replace with actual values)
# databricks_instance = "<DATABRICKS_INSTANCE>"  # e.g., "adb-123456789.azuredatabricks.net"
# job_id = "<DATABRICKS_JOB_ID>"                 # Your Databricks job ID
# token = "<DATABRICKS_TOKEN>"                    # From Key Vault or secure storage
# 
# run_id = trigger_databricks_job(databricks_instance, job_id, token)

---

## 📝 Summary

This notebook demonstrated comprehensive integration patterns between Azure Databricks and Azure Machine Learning using SDK v2:

### What We Covered:

1. ✅ **Dependency Installation** - azure-ai-ml and azure-identity packages
2. ✅ **Authentication** - DefaultAzureCredential for secure access
3. ✅ **Job Submission** - Running training scripts on AzureML compute
4. ✅ **Model Registration** - Registering models from DBFS to AzureML
5. ✅ **AutoML** - Automated machine learning for classification
6. ✅ **Experiment Tracking** - Logging metrics and parameters with MLflow
7. ✅ **Pipeline Orchestration** - Integrating Databricks jobs in AzureML pipelines

### Best Practices:

- 🔐 **Security**: Never hardcode credentials; use Key Vault or managed identities
- 📊 **Tracking**: Always log experiments for reproducibility
- 🏷️ **Tagging**: Use tags to organize resources and track lineage
- 🔄 **Error Handling**: Implement retry logic for API calls
- 📝 **Documentation**: Document model versions and experiment parameters

### Next Steps:

1. Replace all `<PLACEHOLDER>` values with your actual configuration
2. Create necessary compute clusters in AzureML workspace
3. Set up data assets and MLTable datasets
4. Configure service principal or managed identity authentication
5. Test each section independently before running end-to-end

### Resources:

- [Azure ML SDK v2 Documentation](https://learn.microsoft.com/azure/machine-learning/)
- [Databricks-AzureML Integration Guide](https://learn.microsoft.com/azure/databricks/)
- [MLflow Tracking](https://www.mlflow.org/docs/latest/tracking.html)
- [AutoML in Azure ML](https://learn.microsoft.com/azure/machine-learning/concept-automated-ml)

---

**Created**: 2026-02-18  
**SDK Version**: azure-ai-ml 1.x  
**Compatible with**: Azure ML SDK v2, Databricks Runtime 13.x+